In [22]:
!pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
!pip install groq
import os
from groq import Groq
client = Groq(
    api_key = os.getenv("GROQ_API_KEY")
)

In [24]:
models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound-mini
llama-3.1-8b-instant
whisper-large-v3
llama-3.3-70b-versatile
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
groq/compound
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
whisper-large-v3-turbo
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english


In [25]:
def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [26]:
llm('hey whats up?')

"Not much. What's on your mind?"

In [27]:
question = "can i join the course i just discovered?"
answer = llm(question)
print(answer)


I'd be happy to try and help you with that, but I need more information about the course you're interested in. Could you please provide me with more details such as:

1. What is the course about?
2. Who is offering the course?
3. When does the course start (if there's a specific start date)?
4. Do you meet the course requirements or prerequisites?
5. What's the application process (if any)?

This information will help me give you a more accurate answer about your chances of joining the course.


In [28]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [29]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [30]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
can i join the course i just discovered?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate 

In [31]:
answer = llm(prompt)
print(answer)

You can join the course you just discovered, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [ ]:
#we need to make this 
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [33]:
!pip install requests
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [34]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [35]:
documents[500]

{'id': '833fbaa15f',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What are the prerequisites and system requirements for this course?',
 'answer': "You don't need any previous experience with AI tools. A basic ability to program (Python, JavaScript, or similar) is enough to follow the materials and complete the projects."}

In [36]:
from minsearch import Index
index = Index(
    text_fields = ['question', 'section', 'answer'],
    keyword_fields = ['course']
)
index.fit(documents)

In [ ]:
search_results = index.search(question, boost_dict = {"question": 2, "section": 0.5}, filter_dict = {"course": "llm-zoomcamp"}, num_results = 5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [ ]:
#boosting a field in the search results 

In [39]:
def search(question, course = "llm-zoomcamp"):
    return index.search(question, boost_dict = {"question": 2, "section": 0.5}, filter_dict = {"course": course}, num_results = 5)

In [40]:
search_results = search(question)

In [41]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [43]:
INSTRUCTION = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [44]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [45]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [46]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [47]:
USER_PROMPT = USER_PROMPT_TEMPLATE.format(question = question, context = context)
print(USER_PROMPT)


Question:
can i join the course i just discovered?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

Y

In [48]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question = question, context = context)
    return prompt.strip()

In [49]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
can i join the course i just discovered?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

Yo

In [50]:

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)


In [51]:
response.choices[0].message.content

'Based on the given context, it appears that you can join the course you just discovered, but receiving a certificate might depend on various conditions.\n\nIf you want to receive a certificate, you must submit your project while the submission form is still open. Additionally, the course seems to offer a "live" cohort mode and a self-paced mode, but the self-paced mode is not eligible for receiving a certificate since peer-review is a required component.\n\nConsidering these details, the answer is:\n\nYes, you can join the course, but you should be aware of the conditions for receiving the certificate and plan accordingly to submit your project while the submission window is open.'

In [52]:
print(response.model_dump_json(indent = 2))

{
  "id": "chatcmpl-43fb840a-c2ac-42e3-9113-909811ab3bef",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Based on the given context, it appears that you can join the course you just discovered, but receiving a certificate might depend on various conditions.\n\nIf you want to receive a certificate, you must submit your project while the submission form is still open. Additionally, the course seems to offer a \"live\" cohort mode and a self-paced mode, but the self-paced mode is not eligible for receiving a certificate since peer-review is a required component.\n\nConsidering these details, the answer is:\n\nYes, you can join the course, but you should be aware of the conditions for receiving the certificate and plan accordingly to submit your project while the submission window is open.",
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call":

In [58]:
response.choices[0].message.content

'Based on the given context, it appears that you can join the course you just discovered, but receiving a certificate might depend on various conditions.\n\nIf you want to receive a certificate, you must submit your project while the submission form is still open. Additionally, the course seems to offer a "live" cohort mode and a self-paced mode, but the self-paced mode is not eligible for receiving a certificate since peer-review is a required component.\n\nConsidering these details, the answer is:\n\nYes, you can join the course, but you should be aware of the conditions for receiving the certificate and plan accordingly to submit your project while the submission window is open.'

In [59]:
response.usage

CompletionUsage(completion_tokens=131, prompt_tokens=364, total_tokens=495, completion_time=0.235032555, completion_tokens_details=None, prompt_time=0.022146807, prompt_tokens_details=None, queue_time=0.162466323, total_time=0.257179362)

In [61]:
message_history = [
    #SYSTEM/DEVELOPER PROMPT IS CONSTANT ACROSS THE CONV BUT THE USER PROMPT CHANGES BASED ON THE QUESTION AND SEARCH RESULTS
    {"role": "system", "content": INSTRUCTION},
    {"role": "user", "content": prompt}
]
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages= message_history
)

In [63]:
def llm(instructions, user_prompt, model="llama-3.1-8b-instant"):
    message_history = [
        {"role": "developer", "content": INSTRUCTION},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [ ]:

def rag(query, model="llama-3.1-8b-instant"):
    search_results = search(query)
    user_prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTION, user_prompt, model=model)
    return answer

In [65]:
answer = rag(question)
print(answer)

Yes, you can join the course, even if you just discovered it.


In [66]:
rag("How do I get a certificate?")

'Based on the provided context, to get a certificate, you need to:\n\n1. **Finish the course with a "live" cohort**: You can only get a certificate if you finish the course with a live cohort, not in self-paced mode.\n2. **Pass the Capstone project**: You need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n3. **Peer-review 3 capstone projects**: As per the course requirements, you need to peer-review 3 capstone projects after submitting your project.\n4. **Set up your profile correctly**: Make sure to set up your course profile correctly, including filling in your official name as in your identification documents (passport, national ID card, driver\'s license, etc.), to avoid having "Lucid Elbakyan" appear on your certificate.'